# Transform Customer Data
1. Remove records with NULL customer_id
2. Remove exact duplicate records
3. Remove duplicate records based on created_timestamp
4. CAST the columns to the correct Data Type
5. Write transformed data to the Silver schema

## 1. Remove records with NULL customer_id

In [0]:
df = spark.table('gizmobox.bronze.v_customers')
display(df)

## 2. Remove exact duplicate records

In [0]:
df_without_null_customer_id = df.filter(df.customer_id.isNotNull())
display(df_without_null_customer_id)

In [0]:
from pyspark.sql.functions import col

df_without_null_customer_id = df.where(col('customer_id').isNotNull())
display(df_without_null_customer_id)

In [0]:
distinct_df = df_without_null_customer_id.dropDuplicates()
display(distinct_df)

In [0]:
distinct_df = df_without_null_customer_id.distinct()
display(distinct_df)

## 3. Remove duplicate records based on created_timestamp

In [0]:
display(distinct_df.orderBy(col('customer_id')))

In [0]:
from pyspark.sql import functions as F
df_max_ts = distinct_df.groupBy("customer_id") \
        .agg(F.max('created_timestamp').alias("max_created_timestamp"))

display(df_max_ts)

In [0]:
df_distinct_customer = (
    distinct_df.join(df_max_ts, (distinct_df.customer_id == df_max_ts.customer_id) 
                     & (distinct_df.created_timestamp == df_max_ts.max_created_timestamp),
                     'inner')
    .select(distinct_df['*'])
)

display(df_distinct_customer)

## 4. CAST the columns to the correct Data Type

In [0]:
df_casted_customer = (
    df_distinct_customer
        .select(
            df_distinct_customer.created_timestamp.cast('timestamp'),
            df_distinct_customer.customer_id,
            df_distinct_customer.customer_name,
            df_distinct_customer.date_of_birth.cast('date'),
            df_distinct_customer.email,
            df_distinct_customer.member_since.cast('date'),
            df_distinct_customer.telephone
    )
)

display(df_casted_customer)

In [0]:
df_casted_customer = (
    df_distinct_customer
    .withColumn('customer_id', df_distinct_customer.customer_id.cast('integer'))
    .withColumn('customer_name', df_distinct_customer.customer_name.cast('string'))
    .withColumn('email', df_distinct_customer.email.cast('string'))
    .withColumn('telephone', df_distinct_customer.telephone.cast('string'))
    .withColumn('created_timestamp', df_distinct_customer.created_timestamp.cast('timestamp'))
    .withColumn('date_of_birth', df_distinct_customer.date_of_birth.cast('date'))
    .withColumn('member_since', df_distinct_customer.member_since.cast('date'))
)
    
display( df_casted_customer )

## 5. Write transformed data to the Silver schema

In [0]:
df_casted_customer.writeTo("gizmobox.silver.customers").createOrReplace()

In [0]:
df = spark.table("gizmobox.silver.customers")
display(df)